# CHAPTER 10 - Data Aggregation and Group Operations

----

In [1]:
%run "Ch00 - Basic Imports and Set ups.ipynb"

Found in: .\Ch00 - Basic Imports and Set ups.ipynb (Cell #4)
Found in: .\Ch06 - Data Loading, Storage, and File Formats.ipynb (Cell #208)
Found in: .\Ch06 - Data Loading, Storage, and File Formats.ipynb (Cell #219)
Found in: .\Ch06 - Data Loading, Storage, and File Formats.ipynb (Cell #220)
Found in: .\forgitandbinder\PyData-Analysis-3E-Wes-McKinney\Ch00 - Basic Imports and Set ups.ipynb (Cell #4)
Found in: .\forgitandbinder\PyData-Analysis-3E-Wes-McKinney\Ch06 - Data Loading, Storage, and File Formats.ipynb (Cell #208)
Found in: .\forgitandbinder\PyData-Analysis-3E-Wes-McKinney\Ch06 - Data Loading, Storage, and File Formats.ipynb (Cell #219)
Found in: .\forgitandbinder\PyData-Analysis-3E-Wes-McKinney\Ch06 - Data Loading, Storage, and File Formats.ipynb (Cell #220)


Categorizing a dataset and applying a function to each group, whether an aggregation
or transformation, is often a critical component of a data analysis workflow. After
loading, merging, and preparing a dataset, you may need to compute group statistics
or possibly pivot tables for reporting or visualization purposes. pandas provides a
flexible `groupby` interface, enabling you to slice, dice, and summarize datasets in a
natural way.

One reason for the popularity of relational databases and SQL (which stands for
“structured query language”) is the ease with which data can be joined, filtered, transformed,
and aggregated. However, query languages like SQL are somewhat constrained
in the kinds of group operations that can be performed. As you will see, with
the expressiveness of Python and pandas, we can perform quite complex group operations
by utilizing any function that accepts a pandas object or NumPy array. In this
chapter, you will learn how to:

    • Split a pandas object into pieces using one or more keys (in the form of functions, arrays, or DataFrame column names)
    • Calculate group summary statistics, like count, mean, or standard deviation, or a user-defined function    
    • Apply within-group transformations or other manipulations, like normalization, linear regression, rank, or subset selection
    • Compute pivot tables and cross-tabulations
    • Perform quantile analysis and other statistical group analyses

----

Aggregation of time series data, a special use case of groupby, is
referred to as resampling in this book and will receive separate
treatment in Chapter 11.

## Basic Imports and Set ups

In [2]:
%matplotlib inline
import pandas as pd
import numpy as np
import os
from faker import Faker
import matplotlib.pyplot as plt
from datetime import datetime
from IPython.display import display, HTML
import seaborn as sns

In [3]:
import pandas as pd
from IPython.display import display, HTML

def render_book_table(title, columns, rows):
    """
    Render a documentation-style table in Jupyter.

    Parameters
    ----------
    title : str
        Title displayed above the table
    columns : list
        Column names
    rows : list of lists
        Table data rows
    """

    table_data = pd.DataFrame(rows, columns=columns)

    styled_table = (
        table_data.style
        .hide(axis="index")
        .set_table_styles([
            {"selector": "th",
             "props": [("background-color", "#f2f2f2"),
                       ("font-weight", "bold"),
                       ("text-align", "center"),
                       ("border", "1px solid #999"),
                       ("padding", "8px")]},

            {"selector": "td",
             "props": [("border", "1px solid #999"),
                       ("padding", "8px"),
                       ("white-space", "normal"),
                       ("text-align", "left")]},

            {"selector": "table",
             "props": [("border-collapse", "collapse"),
                       ("width", "100%")]}
        ])
    )

    display(HTML(f"<h3>{title}</h3>"))
    display(styled_table)



## 10.1 GroupBy Mechanics

Hadley Wickham, an author of many popular packages for the R programming language,
coined the term split-apply-combine for describing group operations. In the
first stage of the process, data contained in a pandas object, whether a Series, Data‐
Frame, or otherwise, is split into groups based on one or more keys that you provide.
The splitting is performed on a particular axis of an object. For example, a DataFrame
can be grouped on its rows (`axis=0`) or its columns (`axis=1`). Once this is done, a
function is applied to each group, producing a new value. Finally, the results of all
those function applications are combined into a result object. The form of the resulting
object will usually depend on what’s being done to the data. See Figure BELOW for a
mockup of a simple group aggregation.

<img src="images/ch10-1-group.jpg" width="500" alt="Figure 10-1. Illustration of a group aggregation" />

Each grouping key can take many forms, and the keys do not have to be all of the
same type:

    • A list or array of values that is the same length as the axis being grouped
    • A value indicating a column name in a DataFrame
    • A dict or Series giving a correspondence between the values on the axis being grouped and the group names
    • A function to be invoked on the axis index or the individual labels in the index

Note that the latter three methods are shortcuts for producing an array of values to be
used to split up the object. Don’t worry if this all seems abstract. Throughout this
chapter, I will give many examples of all these methods. To get started, here is a small
tabular dataset as a DataFrame:

In [4]:
df = pd.DataFrame({'key1' : ['a', 'a', 'b', 'b', 'a'],
                   'key2' : ['one', 'two', 'one', 'two', 'one'],
                   'data1' : np.random.randn(5),
                   'data2' : np.random.randn(5)})

In [5]:
df

,key1,key2,data1,data2
0,a,one,-0.657443,-0.788235
1,a,two,1.282388,1.573464
2,b,one,0.193691,-0.608151
3,b,two,1.140219,-2.649528
4,a,one,-0.032972,-0.844257


Suppose you wanted to compute the mean of the data1 column using the labels from
`key1`. There are a number of ways to do this. One is to access data1 and call groupby
with the column (a Series) at `key1`:

In [6]:
grouped = df['data1'].groupby(df['key1'])

In [7]:
grouped

This grouped variable is now a GroupBy object. It has not actually computed anything
yet except for some intermediate data about the group key `df['key1']`. The idea is
that this object has all of the information needed to then apply some operation to
each of the groups. For example, to compute group means we can call the GroupBy’s
`mean` method:

In [8]:
grouped.mean()

key1
a    0.197324
b    0.666955
Name: data1, dtype: float64

Later, I’ll explain more about what happens when you call `.mean()`. The important
thing here is that the data (a Series) has been aggregated according to the group key,
producing a new Series that is now indexed by the unique values in the `key1` column.

The result index has the name `'key1'` because the DataFrame column `df['key1']`
did.

If instead we had passed multiple arrays as a list, we’d get something different:

In [9]:
means = df['data1'].groupby([df['key1'], df['key2']]).mean()

In [10]:
means

key1  key2
a     one    -0.345208
      two     1.282388
b     one     0.193691
      two     1.140219
Name: data1, dtype: float64

Here we grouped the data using two keys, and the resulting Series now has a hierarchical
index consisting of the unique pairs of keys observed:

In [11]:
means.unstack()

key2,one,two
key1,,
a,-0.345208,1.282388
b,0.193691,1.140219


In this example, the group keys are all Series, though they could be any arrays of the
right length:

In [12]:
states = np.array(['Ohio', 'California', 'California', 'Ohio', 'Ohio'])

In [13]:
years = np.array([2005, 2005, 2006, 2005, 2006])

In [14]:
df['data1'].groupby([states, years]).mean()

California  2005    1.282388
            2006    0.193691
Ohio        2005    0.241388
            2006   -0.032972
Name: data1, dtype: float64

Note above , the hierarchical index is build on the group by data set values, not on any column in data frame. 

Frequently the grouping information is found in the same DataFrame as the data you
want to work on. In that case, you can pass column names (whether those are strings,
numbers, or other Python objects) as the group keys:  ( you dont have to use full `dataframe[<column name>]` format)



In [15]:
df

,key1,key2,data1,data2
0,a,one,-0.657443,-0.788235
1,a,two,1.282388,1.573464
2,b,one,0.193691,-0.608151
3,b,two,1.140219,-2.649528
4,a,one,-0.032972,-0.844257


In [16]:
df.groupby('key1').mean()

TypeError: agg function failed [how->mean,dtype->object]

In [ ]:
df.groupby('key1').mean(numeric_only=True)

##### ✅ Root Cause of This Pandas Error (Very Important Concept)

This error is **NOT a bug in pandas** 🙂
It is a **data-type problem during aggregation.**

---

##### ⭐ What the Error Really Says

The key line is:

```
TypeError: Could not convert string 'onetwoone' to numeric
```

And later:

```
agg function failed [how->mean, dtype->object]
```

This means:

👉 You are doing:

```python
df.groupby('key1').mean()
```

But **at least one column inside `df` is of type `object` (string)**
and pandas is trying to compute **mean on that column.**

---

##### ⭐ Why Pandas Is Trying To Do Mean On Strings

In modern pandas versions:

When you run:

```python
df.groupby('key1').mean()
```

pandas tries to aggregate **ALL columns** unless told otherwise.

If a column contains values like:

```
one
two
one
```

Then internally pandas tries something like:

```
"one" + "two" + "one"
→ "onetwoone"
```

Then it tries to convert that into numeric → 💥 ERROR.

---

##### ⭐ Example That Produces Same Error

```python
df = pd.DataFrame({
    'key1': ['a','a','b'],
    'val1': [1,2,3],
    'val2': ['one','two','one']
})

df.groupby('key1').mean()
```

❌ Boom → same error.

---

##### ✅ Correct Ways To Fix It (Senior DS Level Understanding)

##### ⭐ Fix 1 — Select Only Numeric Columns (Best Practice)

This is the **cleanest real-world solution.**

```python
df.groupby('key1')[['val1']].mean()
```

OR

```python
df.groupby('key1').mean(numeric_only=True)
```

---

##### ⭐ Fix 2 — Convert Column To Numeric (If It Should Be Numeric)

If column is wrongly typed:

```python
df['val2'] = pd.to_numeric(df['val2'], errors='coerce')
```

---

##### ⭐ Fix 3 — Use Different Aggregation For Strings

Sometimes in DS we WANT string aggregation.

Example:

```python
df.groupby('key1').agg({
    'val1': 'mean',
    'val2': 'first'
})
```

---

##### ⭐ VERY Important Data Science Insight (Interview Level)

##### 🔥 GroupBy Aggregations Require Semantic Meaning

You must always ask:

> “Does mean make sense for this column?”

| Column Type     | Valid Aggregation  |
| --------------- | ------------------ |
| numeric         | mean, sum, std     |
| category/string | count, first, mode |
| datetime        | min, max           |

This is a **core Data Science maturity signal.**

---



You may have noticed in the first case `df.groupby('key1').mean()` that there is no
`key2` column in the result. Because `df['key2']` is not numeric data, it is said to be a
nuisance column, which is therefore excluded from the result. By default, all of the
numeric columns are aggregated, though it is possible to filter down to a subset, as
you’ll see soon.
Regardless of the objective in using groupby, a generally useful GroupBy method is
size, which returns a Series containing group sizes:

# 🎯
> Note that, at the time of this notebook (Mar 2026), the nuisance columns can be removed only if you use `numeric_only=True`

Regardless of the objective in using groupby, a generally useful GroupBy method is
`size`, which returns a Series containing group sizes:

In [ ]:
df.groupby(['key1', 'key2']).size()

In [ ]:
df.groupby('key1').size()

Take note that any missing values in a group key will be excluded from the result.

---

### Iterating Over Groups

The GroupBy object supports iteration, generating a sequence of 2-tuples containing
the group name along with the chunk of data. Consider the following:

In [17]:
for name, group in df.groupby('key1'):
    print(name)
    print(group)

a
  key1 key2     data1     data2
0    a  one -0.657443 -0.788235
1    a  two  1.282388  1.573464
4    a  one -0.032972 -0.844257
b
  key1 key2     data1     data2
2    b  one  0.193691 -0.608151
3    b  two  1.140219 -2.649528


In the case of multiple keys, the first element in the tuple will be a tuple of key values:

In [18]:
for (name1,name2), group in df.groupby(['key1', 'key2']) :
    print(name1,name2)
    print(group)

a one
  key1 key2     data1     data2
0    a  one -0.657443 -0.788235
4    a  one -0.032972 -0.844257
a two
  key1 key2     data1     data2
1    a  two  1.282388  1.573464
b one
  key1 key2     data1     data2
2    b  one  0.193691 -0.608151
b two
  key1 key2     data1     data2
3    b  two  1.140219 -2.649528


Of course, you can choose to do whatever you want with the pieces of data. A recipe
you may find useful is computing a dict of the data pieces as a one-liner:

In [19]:
pieces = dict(list(df.groupby('key1')))

In [20]:
pieces

{'a':   key1 key2     data1     data2
 0    a  one -0.657443 -0.788235
 1    a  two  1.282388  1.573464
 4    a  one -0.032972 -0.844257,
 'b':   key1 key2     data1     data2
 2    b  one  0.193691 -0.608151
 3    b  two  1.140219 -2.649528}

In [21]:
pieces['b']

,key1,key2,data1,data2
2,b,one,0.193691,-0.608151
3,b,two,1.140219,-2.649528


(a dict of collection of data frames group ed by columns of your interest)

By default `groupby` groups on `axis=0`, but you can group on any of the other axes.
For example, we could group the columns of our example `df` here by `dtype` like so:

In [22]:
df.dtypes

key1      object
key2      object
data1    float64
data2    float64
dtype: object

In [23]:
df

,key1,key2,data1,data2
0,a,one,-0.657443,-0.788235
1,a,two,1.282388,1.573464
2,b,one,0.193691,-0.608151
3,b,two,1.140219,-2.649528
4,a,one,-0.032972,-0.844257


In [24]:
grouped = df.groupby(df.dtypes, axis=1)

C:\Users\212364780.HCAD\AppData\Local\Temp\ipykernel_22632\521057107.py:1: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  grouped = df.groupby(df.dtypes, axis=1)


In [25]:
for dtype, group in grouped:
    print(dtype)
    print(group)

float64
      data1     data2
0 -0.657443 -0.788235
1  1.282388  1.573464
2  0.193691 -0.608151
3  1.140219 -2.649528
4 -0.032972 -0.844257
object
  key1 key2
0    a  one
1    a  two
2    b  one
3    b  two
4    a  one


(this can be used to segragate various kind of data types in a frame).

Another way to do the same ( Above one gave deprecation warning).

In [26]:
grouped = df.T.groupby(df.dtypes)

for dtype, group in grouped:
    print(dtype)
    print(group.T)

float64
      data1     data2
0 -0.657443 -0.788235
1  1.282388  1.573464
2  0.193691 -0.608151
3  1.140219 -2.649528
4 -0.032972 -0.844257
object
  key1 key2
0    a  one
1    a  two
2    b  one
3    b  two
4    a  one


##### ✅ Why You Are Getting This Warning

You wrote:

```python
grouped = df.groupby(df.dtypes, axis=1)
```

This means:

👉 **Group columns (NOT rows)** based on their dtype.

Earlier pandas allowed:

```
groupby(..., axis=1)
```

But now this is **deprecated**.

So pandas shows warning:

```
FutureWarning: DataFrame.groupby with axis=1 is deprecated.
Do frame.T.groupby(...) without axis instead.
```

---

##### ✅ Correct Modern Usage (Very Important)

You must **transpose first → then group normally.**

```python
grouped = df.T.groupby(df.dtypes)
```

---

##### ⭐ Why This Works

Original intention:

> Group columns by dtype

But pandas wants **groupby to always work row-wise (axis=0)**

So we convert:

| Step                 | Meaning                                        |
| -------------------- | ---------------------------------------------- |
| `df.T`               | Columns become rows                            |
| `groupby(df.dtypes)` | Now grouping those “rows” (= original columns) |

---

##### ⭐ Full Example (Your Case)

```python
grouped = df.T.groupby(df.dtypes)

for dtype, group in grouped:
    print(dtype)
    print(group)
```

---

##### ⭐ Even Cleaner Senior DS Style

Usually we also store dtype mapping first:

```python
col_types = df.dtypes

grouped = df.T.groupby(col_types)
```

Very readable in production notebooks.

---

##### ⭐ What This Grouping Actually Produces

Your dataframe has:

| Column | dtype  |
| ------ | ------ |
| key1   | object |
| key2   | object |
| data1  | float  |
| data2  | float  |

So grouping result becomes:

```
object → key1, key2
float64 → data1, data2
```

This is extremely useful for:

✅ numeric preprocessing
✅ categorical encoding
✅ automated pipelines

---

##### 🚀 Senior Architect Insight

Modern pandas direction is:

> ❗ Remove axis parameter from many APIs
> ❗ Force consistent row-wise mental model

So future-safe patterns are:

| Old                    | New                 |
| ---------------------- | ------------------- |
| `groupby(..., axis=1)` | `df.T.groupby(...)` |
| `sum(axis=1)`          | still OK            |
| `apply(axis=1)`        | still OK            |

But **groupby(axis=1) → going away.**

---



### Selecting a Column or Subset of Columns

In [27]:
df

,key1,key2,data1,data2
0,a,one,-0.657443,-0.788235
1,a,two,1.282388,1.573464
2,b,one,0.193691,-0.608151
3,b,two,1.140219,-2.649528
4,a,one,-0.032972,-0.844257


In [28]:
df['data1']

0   -0.657443
1    1.282388
2    0.193691
3    1.140219
4   -0.032972
Name: data1, dtype: float64

In [29]:
df[['data2']]

,data2
0,-0.788235
1,1.573464
2,-0.608151
3,-2.649528
4,-0.844257


In [31]:
df[['data1', 'data2']]

,data1,data2
0,-0.657443,-0.788235
1,1.282388,1.573464
2,0.193691,-0.608151
3,1.140219,-2.649528
4,-0.032972,-0.844257


Indexing a GroupBy object created from a DataFrame with a column name or array
of column names has the effect of column subsetting for aggregation. This means
that:

In [32]:
df.groupby('key1')['data1']

is same as 

In [33]:
df['data1'].groupby(df['key1'])

In [34]:
df.groupby('key1')[['data2']]

is same as

In [35]:
df[['data2']].groupby(df['key1'])

Especially for large datasets, it may be desirable to aggregate only a few columns. For
example, in the preceding dataset, to compute means for just the `data2` column and
get the result as a DataFrame, we could write:

In [36]:
df.groupby(['key1', 'key2'])[['data2']].mean()

data2
key1 key2          
a    one  -0.816246
     two   1.573464
b    one  -0.608151
     two  -2.649528

The object returned by this indexing operation is a grouped DataFrame if a list or
array is passed or a grouped Series if only a single column name is passed as a scalar:

In [37]:
s_grouped = df.groupby(['key1', 'key2'])['data2']

In [38]:
s_grouped

In [39]:
s_grouped.mean()

key1  key2
a     one    -0.816246
      two     1.573464
b     one    -0.608151
      two    -2.649528
Name: data2, dtype: float64

---

### Grouping with Dicts and Series

Grouping information may exist in a form other than an array. Let’s consider another
example DataFrame:

In [40]:
people = pd.DataFrame(np.random.randn(5, 5),
                      columns=['a', 'b', 'c', 'd', 'e'],
                      index=['Joe', 'Steve', 'Wes', 'Jim', 'Travis'])

In [41]:
people.iloc[2:3, [1, 2]] = np.nan

In [42]:
people

,a,b,c,d,e
Joe,-1.377362,-1.467437,2.371222,-1.773167,-2.125964
Steve,-0.445317,-0.055088,-0.324728,1.526595,0.851903
Wes,-1.134903,NaN,NaN,-0.732003,1.283313
Jim,-0.060313,0.006152,0.116543,-0.387570,0.383990
Travis,-0.188403,1.082705,-0.587520,1.964299,-0.572058


Now, suppose I have a group correspondence for the columns and want to sum
together the columns by group:

In [43]:
mapping = {'a': 'red', 'b': 'red', 'c': 'blue','d': 'blue', 'e': 'red', 'f' : 'orange'}

In [44]:
mapping

{'a': 'red', 'b': 'red', 'c': 'blue', 'd': 'blue', 'e': 'red', 'f': 'orange'}

Now, you could construct an array from this dict to pass to groupby, but instead we
can just pass the dict (I included the key `'f'` to highlight that unused grouping keys
are OK):

In [46]:
by_column = people.groupby(mapping, axis=1)

C:\Users\212364780.HCAD\AppData\Local\Temp\ipykernel_22632\3111365124.py:1: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  by_column = people.groupby(mapping, axis=1)


In [47]:
by_column.sum()

,blue,red
Joe,0.598056,-4.970763
Steve,1.201867,0.351499
Wes,-0.732003,0.148410
Jim,-0.271026,0.329829
Travis,1.376779,0.322244


The same functionality holds for Series, which can be viewed as a fixed-size mapping:

In [49]:
map_series = pd.Series(mapping)

In [50]:
people.groupby(map_series, axis=1).count()

C:\Users\212364780.HCAD\AppData\Local\Temp\ipykernel_22632\1833935060.py:1: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  people.groupby(map_series, axis=1).count()


,blue,red
Joe,2,3
Steve,2,3
Wes,1,2
Jim,2,3
Travis,2,3


### Grouping with Functions

Using Python functions is a more generic way of defining a group mapping compared
with a dict or Series. Any function passed as a group key will be called once per index
value, with the return values being used as the group names. More concretely, consider
the example DataFrame from the previous section, which has people’s first
names as index values. Suppose you wanted to group by the length of the names;
while you could compute an array of string lengths, it’s simpler to just pass the len
function:

In [51]:
people

,a,b,c,d,e
Joe,-1.377362,-1.467437,2.371222,-1.773167,-2.125964
Steve,-0.445317,-0.055088,-0.324728,1.526595,0.851903
Wes,-1.134903,NaN,NaN,-0.732003,1.283313
Jim,-0.060313,0.006152,0.116543,-0.387570,0.383990
Travis,-0.188403,1.082705,-0.587520,1.964299,-0.572058


In [52]:
people.groupby(len).sum()

,a,b,c,d,e
3,-2.572578,-1.461285,2.487765,-2.892739,-0.458661
5,-0.445317,-0.055088,-0.324728,1.526595,0.851903
6,-0.188403,1.082705,-0.587520,1.964299,-0.572058


Mixing functions with arrays, dicts, or Series is not a problem as everything gets converted
to arrays internally:

In [53]:
key_list = ['one', 'one', 'one', 'two', 'two']

In [54]:
people.groupby([len, key_list]).min()

a         b         c         d         e
3 one -1.377362 -1.467437  2.371222 -1.773167 -2.125964
  two -0.060313  0.006152  0.116543 -0.387570  0.383990
5 one -0.445317 -0.055088 -0.324728  1.526595  0.851903
6 two -0.188403  1.082705 -0.587520  1.964299 -0.572058

##### ✅ How `people.groupby([len, key_list]).min()` works

This is actually a **very powerful (and slightly tricky) Pandas feature** 🙂

You are grouping rows using **two grouping keys at the same time**:

1. A **function → `len`**
2. A **list → `key_list`**

Let’s understand step-by-step.

---

## ⭐ Step 1 — Your DataFrame

Index (row labels):

```
Joe
Steve
Wes
Jim
Travis
```

Columns → `a b c d e`

---

## ⭐ Step 2 — First grouping key → `len`

When you pass a **function inside `groupby()`**, Pandas applies that function to the **index values**.

So:

```
len("Joe")     → 3
len("Steve")   → 5
len("Wes")     → 3
len("Jim")     → 3
len("Travis")  → 6
```

So first grouping key becomes:

```
[3, 5, 3, 3, 6]
```

---

## ⭐ Step 3 — Second grouping key → `key_list`

You manually gave:

```
key_list = ['one', 'one', 'one', 'two', 'two']
```

So second grouping key:

```
['one', 'one', 'one', 'two', 'two']
```

---

## ⭐ Step 4 — Pandas combines both keys

Pandas creates **Multi-level grouping (hierarchical grouping)**.

Each row gets a grouping label like:

| Person | len | key_list | Final Group |
| ------ | --- | -------- | ----------- |
| Joe    | 3   | one      | (3, one)    |
| Steve  | 5   | one      | (5, one)    |
| Wes    | 3   | one      | (3, one)    |
| Jim    | 3   | two      | (3, two)    |
| Travis | 6   | two      | (6, two)    |

---

## ⭐ Step 5 — Then `.min()` is applied

Now Pandas computes **column-wise minimum inside each group.**

So effectively:

### Group (3, one) → rows: Joe, Wes

Min is computed column-wise.

### Group (5, one) → row: Steve

Only one row → values stay same.

### Group (3, two) → row: Jim

Same logic.

### Group (6, two) → row: Travis

Same logic.

---

## ⭐ Final Result Structure

You will get a **MultiIndex DataFrame**:

```
             a      b      c      d      e
len key
3   one   ...
    two   ...
5   one   ...
6   two   ...
```

So:

👉 First index level → **length of name**
👉 Second index level → **value from key_list**

---

## 🔥 Very Important Concept (Senior DS Insight)

`groupby()` accepts **ANY of these as grouping keys:**

* column name
* list / array
* Series
* function
* dictionary
* multiple of the above together

This is why:

```python
people.groupby([len, key_list])
```

is valid.

Internally Pandas does:

```
zip(len(index), key_list)
→ build composite group labels
→ split → apply → combine
```

This is classic **Split-Apply-Combine paradigm.**

---

##### ✅ Super Clear Mental Model

Think like this:

> “For each row, compute grouping labels using ALL provided keys,
> then put rows with same labels into one bucket.”

---



## 10.2 Data Aggregation


---

Aggregations refer to any data transformation that produces scalar values from
arrays. The preceding examples have used several of them, including `mean`, `count`,
`min`, and `sum`. You may wonder what is going on when you invoke mean() on a
GroupBy object. Many common aggregations, such as those found in Table 10-1,
have optimized implementations. However, you are not limited to only this set of
methods.

In [56]:
columns = ["Function name", "Description"]

rows = [
["count","Number of non-NA values in the group"],
["sum","Sum of non-NA values"],
["mean","Mean of non-NA values"],
["median","Arithmetic median of non-NA values"],
["std / var","Unbiased (n−1 denominator) standard deviation and variance"],
["min / max","Minimum and maximum of non-NA values"],
["prod","Product of non-NA values"],
["first / last","First and last non-NA values"]
]

render_book_table(
    "Table 10-1. Optimized groupby Methods",
    columns,
    rows
)

Function name,Description
count,Number of non-NA values in the group
sum,Sum of non-NA values
mean,Mean of non-NA values
median,Arithmetic median of non-NA values
std / var,Unbiased (n−1 denominator) standard deviation and variance
min / max,Minimum and maximum of non-NA values
prod,Product of non-NA values
first / last,First and last non-NA values


You can use aggregations of your own devising and additionally call any method that
is also defined on the grouped object. For example, you might recall that `quantile`
computes sample quantiles of a Series or a DataFrame’s columns.

While `quantile` is not explicitly implemented for GroupBy, it is a Series method and
thus available for use. Internally, GroupBy efficiently slices up the Series, calls `piece.quantile(0.9)` for each piece, and then assembles those results together into
the result object:

In [58]:
df

,key1,key2,data1,data2
0,a,one,-0.657443,-0.788235
1,a,two,1.282388,1.573464
2,b,one,0.193691,-0.608151
3,b,two,1.140219,-2.649528
4,a,one,-0.032972,-0.844257


In [59]:
grouped = df.groupby('key1')

In [60]:
grouped['data1'].quantile(0.9)

key1
a    1.019316
b    1.045567
Name: data1, dtype: float64

To use your own aggregation functions, pass any function that aggregates an array to
the aggregate or `agg` method:

In [61]:
def peak_to_peak(arr):
    return arr.max() - arr.min()

In [62]:
grouped.agg(peak_to_peak)

TypeError: unsupported operand type(s) for -: 'str' and 'str'

In [63]:
grouped.agg(peak_to_peak, numeric_only=True)

TypeError: peak_to_peak() got an unexpected keyword argument 'numeric_only'

In [64]:
grouped[['data1','data2']].agg(peak_to_peak)

,data1,data2
key1,,
a,1.939831,2.417721
b,0.946528,2.041376


##### ✅ Brief Explanation

You had:

```python
grouped = df.groupby('key1')

def peak_to_peak(arr):
    return arr.max() - arr.min()
```

---

##### ❌ First Method — `grouped.agg(peak_to_peak)`

This **did not work** because Pandas applied the function to **all columns**, including string columns like `key2`.

So it tried:

```
'two' - 'one'
```

String subtraction is not valid → **TypeError.**

---

##### ❌ Second Method — `numeric_only=True`

This also **did not work** because `numeric_only` is **not a Pandas filtering instruction here.**

Instead, Pandas passed it as an argument to your custom function:

```
peak_to_peak(arr, numeric_only=True)
```

But your function does not accept this parameter → **unexpected keyword argument error.**

---

##### ✅ Working Method

Select numeric columns **before aggregation.**

```python
df.groupby('key1')[['data1','data2']].agg(peak_to_peak)
```

Now the function runs only on numeric data → **no string subtraction → works correctly.**

---

✅ **Key Idea:**
Always control which columns go into `groupby().agg()` when using custom functions.




---

You may notice that some methods like `describe` also work, even though they are not
aggregations, strictly speaking:

In [66]:
grouped.describe()

data1                                                              \
     count      mean       std       min       25%       50%       75%   
key1                                                                     
a      3.0  0.197324  0.990209 -0.657443 -0.345208 -0.032972  0.624708   
b      2.0  0.666955  0.669296  0.193691  0.430323  0.666955  0.903587   

               data2                                                    \
           max count      mean       std       min       25%       50%   
key1                                                                     
a     1.282388   3.0 -0.019676  1.379984 -0.844257 -0.816246 -0.788235   
b     1.140219   2.0 -1.628839  1.443471 -2.649528 -2.139184 -1.628839   

                          
           75%       max  
key1                      
a     0.392614  1.573464  
b    -1.118495 -0.608151

>Custom aggregation functions are generally much slower than the
optimized functions found in Table 10-1. This is because there is
some extra overhead (function calls, data rearrangement) in constructing
the intermediate group data chunks.

### Column-Wise and Multiple Function Application

Let’s return to the tipping dataset from earlier examples. After loading it with
`read_csv`, we add a tipping percentage column `tip_pct`:

In [68]:
tips = pd.read_csv('examples/tips.csv')

In [74]:
tips['tip_pct'] = tips['tip'] / tips['total_bill']

In [75]:
tips[:6]

,total_bill,tip,smoker,day,time,size,tip_pct
0,16.99,1.01,No,Sun,Dinner,2,0.059447
1,10.34,1.66,No,Sun,Dinner,3,0.160542
2,21.01,3.50,No,Sun,Dinner,3,0.166587
3,23.68,3.31,No,Sun,Dinner,2,0.139780
4,24.59,3.61,No,Sun,Dinner,4,0.146808
5,25.29,4.71,No,Sun,Dinner,4,0.186240


As you’ve already seen, aggregating a Series or all of the columns of a DataFrame is a
matter of using `aggregate` with the desired function or calling a method like `mean` or
`std`. However, you may want to aggregate using a different function depending on the
column, or multiple functions at once. Fortunately, this is possible to do, which I’ll
illustrate through a number of examples.

First, I’ll group the tips by day and smoker:

In [76]:
grouped = tips.groupby(['day', 'smoker'])

In [77]:
grouped_pct = grouped['tip_pct']

In [78]:
grouped_pct.agg('mean')

day   smoker
Fri   No        0.151650
      Yes       0.174783
Sat   No        0.158048
      Yes       0.147906
Sun   No        0.160113
      Yes       0.187250
Thur  No        0.160298
      Yes       0.163863
Name: tip_pct, dtype: float64

If you pass a list of functions or function names instead, you get back a DataFrame
with column names taken from the functions:

In [79]:
grouped_pct.agg(['mean', 'std', peak_to_peak])

mean       std  peak_to_peak
day  smoker                                  
Fri  No      0.151650  0.028123      0.067349
     Yes     0.174783  0.051293      0.159925
Sat  No      0.158048  0.039767      0.235193
     Yes     0.147906  0.061375      0.290095
Sun  No      0.160113  0.042347      0.193226
     Yes     0.187250  0.154134      0.644685
Thur No      0.160298  0.038774      0.193350
     Yes     0.163863  0.039389      0.151240

Here we passed a list of aggregation functions to `agg` to evaluate indepedently on the
data groups.

You don’t need to accept the names that GroupBy gives to the columns; notably,
lambda functions have the name `'<lambda>'`, which makes them hard to identify
(you can see for yourself by looking at a function’s `__name__` attribute). Thus, if you
pass a list of `(name, function)` tuples, the first element of each tuple will be used as
the DataFrame column names (you can think of a list of 2-tuples as an ordered
mapping):m

In [83]:
grouped_pct.agg([('foo', 'mean'), ('bar', np.std)])

C:\Users\212364780.HCAD\AppData\Local\Temp\ipykernel_22632\345979284.py:1: FutureWarning: The provided callable <function std at 0x000002988EB10FE0> is currently using SeriesGroupBy.std. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "std" instead.
  grouped_pct.agg([('foo', 'mean'), ('bar', np.std)])


foo       bar
day  smoker                    
Fri  No      0.151650  0.028123
     Yes     0.174783  0.051293
Sat  No      0.158048  0.039767
     Yes     0.147906  0.061375
Sun  No      0.160113  0.042347
     Yes     0.187250  0.154134
Thur No      0.160298  0.038774
     Yes     0.163863  0.039389

With a DataFrame you have more options, as you can specify a list of functions to
apply to all of the columns or different functions per column. To start, suppose we
wanted to compute the same three statistics for the `tip_pct` and `total_bill`
columns:

In [84]:
functions = ['count', 'mean', 'max']

In [85]:
result = grouped['tip_pct', 'total_bill'].agg(functions)

ValueError: Cannot subset columns with a tuple with more than one element. Use a list instead.

In [87]:
result = grouped[['tip_pct', 'total_bill']].agg(functions)

In [88]:
result

tip_pct                     total_bill                  
              count      mean       max      count       mean    max
day  smoker                                                         
Fri  No           4  0.151650  0.187735          4  18.420000  22.75
     Yes         15  0.174783  0.263480         15  16.813333  40.17
Sat  No          45  0.158048  0.291990         45  19.661778  48.33
     Yes         42  0.147906  0.325733         42  21.276667  50.81
Sun  No          57  0.160113  0.252672         57  20.506667  48.17
     Yes         19  0.187250  0.710345         19  24.120000  45.35
Thur No          45  0.160298  0.266312         45  17.113111  41.19
     Yes         17  0.163863  0.241255         17  19.190588  43.11

As you can see, the resulting DataFrame has hierarchical columns, the same as you
would get aggregating each column separately and using `concat` to glue the results
together using the column names as the `keys` argument:

In [90]:
result['tip_pct']

count      mean       max
day  smoker                           
Fri  No          4  0.151650  0.187735
     Yes        15  0.174783  0.263480
Sat  No         45  0.158048  0.291990
     Yes        42  0.147906  0.325733
Sun  No         57  0.160113  0.252672
     Yes        19  0.187250  0.710345
Thur No         45  0.160298  0.266312
     Yes        17  0.163863  0.241255

As before, a list of tuples with custom names can be passed:

In [91]:
ftuples = [('Durchschnitt', 'mean'), ('Abweichung', np.var)]

In [93]:
grouped[['tip_pct', 'total_bill']].agg(ftuples)

C:\Users\212364780.HCAD\AppData\Local\Temp\ipykernel_22632\1836388418.py:1: FutureWarning: The provided callable <function var at 0x000002988EB11120> is currently using SeriesGroupBy.var. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "var" instead.
  grouped[['tip_pct', 'total_bill']].agg(ftuples)


tip_pct              total_bill            
            Durchschnitt Abweichung Durchschnitt  Abweichung
day  smoker                                                 
Fri  No         0.151650   0.000791    18.420000   25.596333
     Yes        0.174783   0.002631    16.813333   82.562438
Sat  No         0.158048   0.001581    19.661778   79.908965
     Yes        0.147906   0.003767    21.276667  101.387535
Sun  No         0.160113   0.001793    20.506667   66.099980
     Yes        0.187250   0.023757    24.120000  109.046044
Thur No         0.160298   0.001503    17.113111   59.625081
     Yes        0.163863   0.001551    19.190588   69.808518

Now, suppose you wanted to apply potentially different functions to one or more of
the columns. To do this, pass a dict to `agg` that contains a mapping of column names
to any of the function specifications listed so far:

In [94]:
grouped.agg({'tip' : np.max, 'size' : 'sum'})

C:\Users\212364780.HCAD\AppData\Local\Temp\ipykernel_22632\683326323.py:1: FutureWarning: The provided callable <function max at 0x000002988EB104A0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped.agg({'tip' : np.max, 'size' : 'sum'})


tip  size
day  smoker             
Fri  No       3.50     9
     Yes      4.73    31
Sat  No       9.00   115
     Yes     10.00   104
Sun  No       6.00   167
     Yes      6.50    49
Thur No       6.70   112
     Yes      5.00    40

In [95]:
grouped.agg({'tip_pct' : ['min', 'max', 'mean', 'std'],'size' : 'sum'})

tip_pct                               size
                  min       max      mean       std  sum
day  smoker                                             
Fri  No      0.120385  0.187735  0.151650  0.028123    9
     Yes     0.103555  0.263480  0.174783  0.051293   31
Sat  No      0.056797  0.291990  0.158048  0.039767  115
     Yes     0.035638  0.325733  0.147906  0.061375  104
Sun  No      0.059447  0.252672  0.160113  0.042347  167
     Yes     0.065660  0.710345  0.187250  0.154134   49
Thur No      0.072961  0.266312  0.160298  0.038774  112
     Yes     0.090014  0.241255  0.163863  0.039389   40

A DataFrame will have hierarchical columns only if multiple functions are applied to
at least one column.

### Returning Aggregated Data Without Row Indexes

In all of the examples up until now, the aggregated data comes back with an index,
potentially hierarchical, composed from the unique group key combinations. Since
this isn’t always desirable, you can disable this behavior in most cases by passing
`as_index=False` to groupby:

In [98]:
tips.groupby(['day', 'smoker'], as_index=False).mean()

TypeError: agg function failed [how->mean,dtype->object]

In [99]:
tips.groupby(['day', 'smoker'], as_index=False).mean(numeric_only=True)

,day,smoker,total_bill,tip,size,tip_pct
0,Fri,No,18.420000,2.812500,2.250000,0.151650
1,Fri,Yes,16.813333,2.714000,2.066667,0.174783
2,Sat,No,19.661778,3.102889,2.555556,0.158048
3,Sat,Yes,21.276667,2.875476,2.476190,0.147906
4,Sun,No,20.506667,3.167895,2.929825,0.160113
5,Sun,Yes,24.120000,3.516842,2.578947,0.187250
6,Thur,No,17.113111,2.673778,2.488889,0.160298
7,Thur,Yes,19.190588,3.030000,2.352941,0.163863


Of course, it’s always possible to obtain the result in this format by calling
`reset_index` on the result. Using the `as_index=False` method avoids some unnecessary
computations.